## Example for extracting data for GPT prompting

### This is not the final/complete code but more about how to get the desired data from the table

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import tiktoken
import openai 
from gpt_cost_estimator import CostEstimator
import os
from openai import AzureOpenAI
import configparser
import json
import time
import pydantic
from pydantic import Field
from typing import Literal, List
from enum import Enum
from pydantic import BaseModel
import outlines
from outlines.models.openai import OpenAI, OpenAIConfig
from pydantic import ValidationError

In [2]:
pd.set_option('display.max_colwidth', None)

# OSA I : andmed failid (n80 580 näidet)

In [3]:
spatial_obl_ex = pd.read_csv("../gpt_input/n80_top29_10p_10n_examples.csv", encoding="utf-8",  sep="|")

In [4]:
def eki_loc(row):
    if row['ekilex_tag'] == 'location':
        return "yes"
    else:
        return "no"

In [5]:
spatial_obl_ex["eki_is_loc"] = spatial_obl_ex.apply(eki_loc, axis=1)

In [6]:
spatial_obl_ex

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase,ekilex_tag,eki_is_loc
0,34594,kodus,kodu,valitseb,valitsema,NaN,in,20101,"Õnneks piisab mulle sellest , kui mina ise ja minu lähedased on terved ning kodus valitseb armastus .",kodus valitseb armastus,location,yes
1,39275,haiglas,haigla,valitses,valitsema,NaN,in,22819,Mustamäe haiglas valitses Bentonite hinnangul tume teadmatus .,haiglas valitses hinnangul teadmatus,location,yes
2,39620,Tööhõiveametis,tööhõiveamet,valitseb,valitsema,NaN,in,23033,"“ Tööhõiveametis valitseb täielik tegematajätmine , ” pragab Raid .",Tööhõiveametis valitseb tegematajätmine,location,yes
3,72824,Muusikakoolis,muusikakool,valitses,valitsema,NaN,in,42060,"“ Muusikakoolis valitses reaalõpetajate terror , nad ei suutnud muusikute hinge mõista .",Muusikakoolis valitses terror,location,yes
4,83937,tööhõiveametis,tööhõiveamet,valitseb,valitsema,NaN,in,48642,Raidi arvates valitseb tööhõiveametis töö korraldamatus .,arvates valitseb tööhõiveametis korraldamatus,location,yes
...,...,...,...,...,...,...,...,...,...,...,...,...
575,1408398,novembris,november,puhkes,puhkema,NaN,in,884383,"Eesti üheks eliitkooliks peetavas Hugo Treffneri gümnaasiumis puhkes tulekahju 1998. aasta novembris , selle tõttu muutus kasutuskõlbmatuks juba varem remonti vajanud hoone Jaani tänava poolne tiib .",gümnaasiumis puhkes tulekahju novembris,time,no
576,1436524,novembris,november,puhkes,puhkema,NaN,in,901833,"1808. aastal lõppes Rootsi ülemvõim ja sama aasta novembris puhkes Helsingis tulekahju , mis hävitas neljandiku linnast ehk 61 maja .",novembris puhkes Helsingis tulekahju,time,no
577,1493775,detsembris,detsember,puhkenud,puhkema,NaN,in,938116,"Valge maja ja teiste riigiasutuste hoonete ümber tugevdati reedel valvet ning võimud lubasid , et ei võimaldada korduda mullu detsembris Seattle'is puhkenud globaliseerumise-vastasel mässul .",detsembris Seattle'is puhkenud,time,no
578,1528461,septembris,september,puhkes,puhkema,NaN,in,959426,""" Eelmise aasta septembris puhkes Kastre metskonnas Aaslava vallas suur põleng , "" rääkis inspektor , kes ei soovinud töö iseloomust tulenevalt oma nime avaldada .",septembris puhkes metskonnas vallas põleng,time,no


# OSA II : GPT

## GPT jaoks vajalik

In [12]:
config = configparser.ConfigParser()

conf_file = '../azure.ini'

status = config.read(conf_file) 
assert status == [conf_file]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [13]:
def in2json(sisend):
    return json.dumps(sisend, ensure_ascii=False)


def user_message(lause, fraas):
    mes ={
            "role": "user",
            "content": {"l": lause, "c": fraas}
            }
    return mes


def assistant_message(yesno, short_ans, long_ans):
    mes = {
            "role": "assistant",
            "content": {"a": yesno, "s": short_ans, "r": long_ans}
            }
    return mes


def messages_2_str(messages:list[dict]):
    # kui tahta samad dict prompti asjad anda ette lihtsalt stringina
    # tekitab 5-realised blokid, eraldatud: \n\n
    
    new_string = ""
    
    for mes in messages:
        if mes["role"]=="user":
            l = mes["content"]["l"]
            c = mes["content"]["c"]
            new_string += f"l: {l}\n"
            new_string += f"c: {c}\n"
        if mes["role"]=="assistant":
            a = mes["content"]["a"]
            s = mes["content"]["s"]
            r = mes["content"]["r"]
            new_string += f"a: {a}\n"
            new_string += f"s: {s}\n"
            new_string += f"r: {r}\n\n"
    
    return new_string

In [111]:
SYSTEM_PROMPT = """
You are a classification assistant.
Your task: Given a list of examples, each with keys "l" (sentence) and "c" (phrase), classify whether "c" is an adverbial of place in the context of the sentence.
Adverbial of place aka location usually answers the question “where”. 
It is a place or concept where something or someone is located, goes to or comes from. 
That place can be concrete (bank, table, Berlin), abstract (literature, soul, government, consciousness, top of a group, concept), inanimate (journal, chair, wifi), alive (mother, Peter, dog), event (dress rehearsal, camp, class, situation).
Locations ARE NOT phrases that show time, state of being, owner, experiencer, instrument, manner OR are purely grammatical constructions. 
If the phrase can answer the question when, in what state, who, with what or how, then it is not a location.  

Analyse the given definition of location and 'few_shots' examples.
Then process the list called 'batch'.

Output JSON requirements:
- Respond strictly with an array of JSON objects, one object per 'batch' item.
- The JSON array must be in the exact same order as the batch items.
- Response must be without markdown or comments.
- Each output JSON must have:
  "a": "yes" (location) or "no" (not location)
"""


In [112]:

FEW_SHOTS = [
            user_message("Me läksime Pariisi.","Pariisi"),
            assistant_message("yes", "city", "City name, answers question 'where'."),
    
            user_message("Ema on mul olnud alati õmblustöö inimene ja õpetab seda praegu ühes õmbluskoolis teistelegi.", "õmbluskoolis"),
            assistant_message("yes", "location", "Organization's building, answers 'where'."),
    
            user_message("Põhjapoolusele saabus kottide-kompsudega tuhandeid võõrtöölisi.", "Põhjapoolusele"),
            assistant_message("yes", "location", "Answers the question 'to where'."),
    
            user_message("Ta tuli idast kõikide oma raamatutega.", "idast"),
            assistant_message("yes", "direction", "Abstract location, answers 'from where'."),
    
            user_message("Mees istus peale pikka päeva uuesti sadulasse.", "sadulasse"),
            assistant_message("yes", "object", "Man sat on an object and answers 'where'."),

            user_message("Nad sõidavad neljapäeval maale.", "neljapäeval"),
            assistant_message("no", "time", "Answers 'when'."),
    
            user_message("Avo süüdistati selles, et ta varastas magava J.P. põuetaskust raha koos rahakotiga.", "põuetaskust"),
            assistant_message("yes", "object", "Answers 'from where' and is an object."),
    
            user_message("Näiteks korraldas Affleck Lopezile üllatus-sünnipäevapeo restoranis Park.", "restoranis"),
            assistant_message("yes", "location", "Physical location and also organization, answers 'where'."),
    
            user_message("HP700 ei ulatu enam SpeedTouchi Wifi'sse.", "Wifi'sse"),
            assistant_message("yes", "abstract", "Abstract location, answers 'where'"),

            user_message("Minnie käis Barbra teadmata isegi kleidiproovis.", "kleidiproovis"),
            assistant_message("yes", "event", "Event that answers 'where' the person was."),

            user_message("Rüselejal käsisid sussid", "Rüselejal"),
            assistant_message("no", "experiencer", "Rüseleja is the experiencer."),
    
            user_message("Nüüd siis istun sitas.", "sitas"),
            assistant_message("no", "state", "State of being."),    
    
            user_message("Protest on mitmekesine ja teravaimalt avaldub see kirjanduses.", "teravaimalt"),
            assistant_message("no", "other", "Verbial of manner."),    
    
            user_message("Korraldasime seminari TTÜs.", "TTÜs"),
            assistant_message("yes", "location", "TTÜ is an organization but in sentence refers to location and answers 'where'."),
    
            user_message("Väga hästi varjab päikesekiiri näiteks markiis.", "markiis"),
            assistant_message("no", "other", "Nominative case and not location."),
    
            user_message("Ingridi puhul läks hiljem täkkesse just see ütelus.", "täkkesse"),
            assistant_message("no", "other", "Phrasal verb and not location."),
    
            user_message("Mari kuulas kikkis kõrvul.", "kõrvul"),
            assistant_message("no", "manner", "Answers the question 'how'."),
    
            user_message("Maril on kaks last.", "Maril"),
            assistant_message("no", "owner", "Answers the question 'who'."),
    
            user_message("See asi ununes mul täielikult.", "mul"),
            assistant_message("no", "experiencer", "Answers the question 'who'."),
    
            user_message("Luba tal ükskord ometi kõik südamelt ära rääkida.", "tal"),
            assistant_message("no", "experiencer", "Answers the question 'who'."),
    
            user_message("Tüdruku nägu on naerul.", "naerul"),
            assistant_message("no", "state", "Answers the question 'in what state'."),
    
            user_message("Munad on vahul.", "vahul"),
            assistant_message("no", "state", "Answers the question 'in what state'."),
    
            user_message("Mari elab juba kolmandat aastat välismaal.", "välismaal"),
            assistant_message("yes", "location", "Answers the question 'where'."),
    
            user_message("Üliõpilased on loengul.", "loengul"),
            assistant_message("yes", "location", "Answers the question 'where'."),
    
            user_message("Müts on peas.", "peas"),
            assistant_message("yes","location", "Answers the question ‘where’."),
    
            user_message("Jüri on Keskerakonnas.", "Keskerakonnas"),
            assistant_message("yes", "location", "Jüri is located in the organization's structure."),
    
            user_message("Tema sünnipäev on märtsis.", "märtsis"),
            assistant_message("no", "time", "Answers question 'when'."),
    
            user_message("Mees on sügavas depressioonis.", "depressioonis"),
            assistant_message("no", "state", "Answers the question 'in what state'."),
    
            user_message("Ta on andekas matemaatikas.", "matemaatikas"),
            assistant_message("no", "construction", "Is purely grammatical."),
    
            user_message("Ma kahtlen teie siiruses.", "siiruses"),
            assistant_message("no", "construction", "Is purely grammatical."),
    
            user_message("Ta mängib orkestris.", "orkestris"),
            assistant_message("yes", "location", "Answers the question 'where'."),
    
            user_message("Rahvamurrus ei leidnud laps ema.", "rahvamurrus"),
            assistant_message("yes", "location", "Answers the question 'where'."),
    
            user_message("Me elame vabaduses, vendluses ja armastuses.", "vabaduses"),
            assistant_message("yes", "location", "Our existence is located in the concept of ‘vabadus’."),
    
            user_message("Sinus on midagi.", "sinus"),
            assistant_message("yes", "location", "Something like a feeling or potential can be located inside of ’sinus’."),
    
            user_message("Maxence märkas ema, hüppas diivanilt püsti ja lülitas televiisori välja, mis äratas emas kohe kahtlusi.", "emas"),
            assistant_message("yes", "location", "kahtlused are located inside of ema. Answers the question ‘where’"),
    
            user_message("Me kasvasime üles botastes.", "botastes"),
            assistant_message("no", "state", "Answers the question 'in what state'."),
    
            user_message("Moos valgus pirukast välja.", "pirukast"),
            assistant_message("yes", "location", "Answers the question 'from where'."),
    
            user_message("Nii voolab riiklikust meditsiinist elujõud muudkui välja .", "meditsiinist"),
            assistant_message("yes", "location", "Conspet, answers the question 'from where'."),
    
            user_message("Uuringu põhjal selgus , et ligi 70 protsenti naistest pöörduks pärast esimese lapse sündi heameelega vanasse töökohta tagasi.", "töökohta"),
            assistant_message("yes", "location", "Answers the question 'to where'."),
    
            user_message("Ema pani Peetrile teki peale.", "Peetrile"),
            assistant_message("yes", "location", "Peeter is not an experiencer in this context, answers the question 'to where'."),
    
            user_message("Toomas Lepp tegutses kaua ETV-s.", "ETV-s"),
            assistant_message("yes", "abstract", "Abstract location, answers 'where'."),
    
            user_message("Ta on lisanud Delfisse mitmeid artikleid.", "Delfisse"),
            assistant_message("yes", "abstract", "Abstract location, answers 'where'."),
    
            user_message("Ta istus hooaja jooksul peatreeneripingile.", "peatreeneripingile"),
            assistant_message("yes", "abstract", "Metaphorical physical movement, answers question 'to where'."),

            user_message("Elu läks rööbastesse tagasi.", "rööbastesse"),
            assistant_message("yes", "abstract", "Metaphorical physical movement, answers question 'to where'."),

            user_message("Organisatsiooni ladvikus on rahu.", "ladvikus"),
            assistant_message("yes", "location", "Peace is a state among the organization's top group, answers question 'where'."),

    
]
 

In [113]:
FEW_SHOTS

[{'role': 'user', 'content': {'l': 'Me läksime Pariisi.', 'c': 'Pariisi'}},
 {'role': 'assistant',
  'content': {'a': 'yes',
   's': 'city',
   'r': "City name, answers question 'where'."}},
 {'role': 'user',
  'content': {'l': 'Ema on mul olnud alati õmblustöö inimene ja õpetab seda praegu ühes õmbluskoolis teistelegi.',
   'c': 'õmbluskoolis'}},
 {'role': 'assistant',
  'content': {'a': 'yes',
   's': 'location',
   'r': "Organization's building, answers 'where'."}},
 {'role': 'user',
  'content': {'l': 'Põhjapoolusele saabus kottide-kompsudega tuhandeid võõrtöölisi.',
   'c': 'Põhjapoolusele'}},
 {'role': 'assistant',
  'content': {'a': 'yes',
   's': 'location',
   'r': "Answers the question 'to where'."}},
 {'role': 'user',
  'content': {'l': 'Ta tuli idast kõikide oma raamatutega.', 'c': 'idast'}},
 {'role': 'assistant',
  'content': {'a': 'yes',
   's': 'direction',
   'r': "Abstract location, answers 'from where'."}},
 {'role': 'user',
  'content': {'l': 'Mees istus peale pikka

In [114]:
FEW_SHOTS_STR = messages_2_str(FEW_SHOTS)

print(FEW_SHOTS_STR)

l: Me läksime Pariisi.
c: Pariisi
a: yes
s: city
r: City name, answers question 'where'.

l: Ema on mul olnud alati õmblustöö inimene ja õpetab seda praegu ühes õmbluskoolis teistelegi.
c: õmbluskoolis
a: yes
s: location
r: Organization's building, answers 'where'.

l: Põhjapoolusele saabus kottide-kompsudega tuhandeid võõrtöölisi.
c: Põhjapoolusele
a: yes
s: location
r: Answers the question 'to where'.

l: Ta tuli idast kõikide oma raamatutega.
c: idast
a: yes
s: direction
r: Abstract location, answers 'from where'.

l: Mees istus peale pikka päeva uuesti sadulasse.
c: sadulasse
a: yes
s: object
r: Man sat on an object and answers 'where'.

l: Nad sõidavad neljapäeval maale.
c: neljapäeval
a: no
s: time
r: Answers 'when'.

l: Avo süüdistati selles, et ta varastas magava J.P. põuetaskust raha koos rahakotiga.
c: põuetaskust
a: yes
s: object
r: Answers 'from where' and is an object.

l: Näiteks korraldas Affleck Lopezile üllatus-sünnipäevapeo restoranis Park.
c: restoranis
a: yes
s: loc

## tokenite arvutuseks

In [ ]:
# example batch

In [19]:
batch_items = []

for i in range(len(spatial_obl_ex)):
    ex = spatial_obl_ex.iloc[i]
    batch_items.append(in2json({"l": ex["sentence"], "c": ex["head_form"]}))

    if i==10:
        break

In [20]:
batch_items

['{"l": "Õnneks piisab mulle sellest , kui mina ise ja minu lähedased on terved ning kodus valitseb armastus .", "c": "kodus"}',
 '{"l": "Mustamäe haiglas valitses Bentonite hinnangul tume teadmatus .", "c": "haiglas"}',
 '{"l": "“ Tööhõiveametis valitseb täielik tegematajätmine , ” pragab Raid .", "c": "Tööhõiveametis"}',
 '{"l": "“ Muusikakoolis valitses reaalõpetajate terror , nad ei suutnud muusikute hinge mõista .", "c": "Muusikakoolis"}',
 '{"l": "Raidi arvates valitseb tööhõiveametis töö korraldamatus .", "c": "tööhõiveametis"}',
 '{"l": "Täpselt niisugune mentaliteet valitses Ukrainas .", "c": "Ukrainas"}',
 '{"l": "Leidmaks lahendust Jaan Tõnissoni kadumisele , tuleks tagasi pöörduda olukorra juurde , mis valitses Tallinnas 1941. aasta juuli alguspäevadel .", "c": "Tallinnas"}',
 '{"l": "Pärast seda kui iseseisvus käes , valitses Eestis mõnd aega aateline rahvuslus , laulva revolutsiooni järgne palang .", "c": "Eestis"}',
 '{"l": "“ Eestis valitseb väga primitiivne arusaam ava

In [23]:
user_payload1 = {
        "few_shots": FEW_SHOTS,
        "batch": batch_items
    }

messages1 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": in2json(user_payload1)}
]

# oletame et mudeli output on sama palju tokeneid kui batch items

print(messages1)

[{'role': 'system', 'content': '\nYou are a classification assistant.\nYour task: Given a list of examples, each with keys "l" (sentence) and "c" (phrase), classify whether "c" is an adverbial of place in the context of the sentence.\nAdverbial of place aka location usually answers the question “where”. \nIt is a place or concept where something or someone is located, goes to or comes from. \nThat place can be concrete (bank, table, Berlin), abstract (literature, soul, government), inanimate (journal, chair, wifi), alive (mother, Peter, dog), event (dress rehearsal, camp, class).\nLocations ARE NOT phrases that show time, state of being, owner, experiencer, instrument, manner OR are purely grammatical constructions. \nIf the phrase can answer the question when, in what state, who, with what or how, then it is not a location.  \n\nAnalyse the given definition of location and \'few_shots\' examples.\nThen process the list called \'batch\'.\n\nOutput JSON requirements:\n- Respond strictly

In [25]:
user_payload2 = {
        "few_shots": FEW_SHOTS_STR,
        "batch": batch_items
    }

messages2 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": in2json(user_payload2)}
]

# oletame et mudeli output on sama palju tokeneid kui batch items

print(messages2)

[{'role': 'system', 'content': '\nYou are a classification assistant.\nYour task: Given a list of examples, each with keys "l" (sentence) and "c" (phrase), classify whether "c" is an adverbial of place in the context of the sentence.\nAdverbial of place aka location usually answers the question “where”. \nIt is a place or concept where something or someone is located, goes to or comes from. \nThat place can be concrete (bank, table, Berlin), abstract (literature, soul, government), inanimate (journal, chair, wifi), alive (mother, Peter, dog), event (dress rehearsal, camp, class).\nLocations ARE NOT phrases that show time, state of being, owner, experiencer, instrument, manner OR are purely grammatical constructions. \nIf the phrase can answer the question when, in what state, who, with what or how, then it is not a location.  \n\nAnalyse the given definition of location and \'few_shots\' examples.\nThen process the list called \'batch\'.\n\nOutput JSON requirements:\n- Respond strictly

### manuaalne umbkaudne sisend

In [26]:
try:
    enc = tiktoken.encoding_for_model("gpt-4o")
except KeyError:
    enc = tiktoken.get_encoding("o200k_base")

def count_tokens(text):
    return len(enc.encode(text))

def count_message_tokens(messages):
    total = 0
    for m in messages:
        total += len(enc.encode(m["role"]))
        total += len(enc.encode(m["content"]))
    return total

In [32]:
print("KOKKU tokeneid:", count_message_tokens(messages1))
print("System prompt tokeneid:", len(enc.encode(SYSTEM_PROMPT)))
#print("Few-shots tokeneid:", count_message_tokens(FEW_SHOTS)) # kui on dict kujul role ja content
print("Few-shots tokeneid:", len(enc.encode(in2json(FEW_SHOTS)))) # kui on lihtsalt stringina
print("väljund tokeneid:", len(enc.encode(" ". join(batch_items))))
print(count_message_tokens(messages1)+len(enc.encode(" ". join(batch_items))))

KOKKU tokeneid: 3892
System prompt tokeneid: 282
Few-shots tokeneid: 3116
väljund tokeneid: 471
4363


In [33]:

print("KOKKU tokeneid:", count_message_tokens(messages2))
print("System prompt tokeneid:", len(enc.encode(SYSTEM_PROMPT)))
#print("Few-shots tokeneid:", count_message_tokens(FEW_SHOTS)) # kui on dict kujul role ja content
print("Few-shots tokeneid:", len(enc.encode(FEW_SHOTS_STR))) # kui on lihtsalt stringina
print("väljund tokeneid:", len(enc.encode(" ". join(batch_items))))
print(count_message_tokens(messages2)+len(enc.encode(" ". join(batch_items))))

KOKKU tokeneid: 2641
System prompt tokeneid: 282
Few-shots tokeneid: 1740
väljund tokeneid: 471
3112


In [72]:
#400*580 # muidu oleks 105070 kirjet 580 asemel

232000

In [34]:
3200*58

185600

### cost estimator

In [43]:
messages3 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": in2json(user_payload2)},
]

In [44]:
@CostEstimator()
def query_openai(model, messages, **kwargs):
    args_to_remove = ['mock', 'completion_tokens']

    for arg in args_to_remove:
        if arg in kwargs:
            del kwargs[arg]

    return openai.ChatCompletion.create(
        model = model,
        messages = messages,
        **kwargs)


responses = []
i = 0
#for i in tqdm(range(0,1)):
response = query_openai(
  model="gpt-4o",
  messages = messages3,
  temperature=0,
  mock=True,
  completion_tokens=1
)

responses.append({
      'input': i,
      'output': response["choices"][0]["message"]["content"]
    })

print() # Empty line to display the total sum

# Print the responses
#print(responses)

Cost: $0.0073 | Total: $0.0073


In [45]:
0.0073*58 # -> eurodes

0.4234

In [46]:
CostEstimator.get_total_cost(CostEstimator)

0.007330000000000001

In [42]:
CostEstimator.reset()

## pydantic

In [57]:

class ClassificationDict(BaseModel):
    a: Literal["yes", "no"]
    #s: str
    #r: str


In [41]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

## Andmete söötmine

In [115]:
def classify_batch(my_batch):

    print("classify", len(my_batch))
    max_att = 2
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "few_shots": FEW_SHOTS_STR,
            "batch": my_batch
        }
    
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": in2json(user_payload)}
        ]

        #print(messages)
        #return None, None
        
        response = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            temperature=0 # absoluutselt min väljund ehk tahan 1 tokenit
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(batch):
                raise ValueError("Väljundis ei ole õige arv vastuseid.")
                
            elif len(data) == len(batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")

    return response, raw_output

In [116]:
df = spatial_obl_ex.iloc[10:20] #.sample(frac=1)#.reset_index(drop=True)
df

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase,ekilex_tag,eki_is_loc
10,122893,veebruaris,veebruar,valitses,valitsema,NaN,in,71692,"1926. aasta veebruaris , kui Helen Wills saabus Cannes'i , et mängida tennist Suzanne Lengleniga , valitses Rivieras postimpressionistlik õhkkond .",veebruaris saabus valitses Rivieras õhkkond,time,no
11,178657,ladvikus,ladvik,valitsenud,valitsema,NaN,in,106005,8. juulil tekkis kaitsejõudude ladvikus valitsenud segaduse tõttu jäägritel vastasseis Vene eriüksusega .,ladvikus valitsenud,not_location,no
12,186045,halduskogudes,halduskogu,valitseb,valitsema,NaN,in,110693,"Ka aedlinnaosade halduskogudes valitseb meelsus , mille järgi tuleks nii põhikrunt kui reservmaa lugeda ühtseks hoone teenindamiseks vajalikuks maaks .",halduskogudes valitseb meelsus,not_location,no
13,304611,suhtumises,suhtumine,valitseb,valitsema,NaN,in,183264,Terav kontrast valitseb ka suhtumises lähimasse läänenaabrisse .,kontrast valitseb suhtumises,not_location,no
14,347538,teadvuses,teadvus,valitses,valitsema,NaN,in,213063,"Kuna vadja keelel puudus aga juba siis väljund ametlikku asjaajamismaailma ( ja tänapäevaks puudub parku juba täielikult igasugune sotsiaalne mõõde ) , valitses Ariste informantide teadvuses valdavalt vene-vadja kaksik- või lausa vene-ingeri-vadja kolmikidentiteet , mis väljaspool Ariste ja ta õpilastega suhtlussituatsioone ka valdas ja igapäevaeluliselt valdama jäi .",puudus valitses teadvuses valdavalt kaksik- kolmikidentiteet,not_location,no
15,692133,koalitsioonis,koalitsioon,valitsesid,valitsema,NaN,in,442427,"CDU kogus absoluutse häälte-enamuse ning hakkab seal nüüd üksi valitsema , seni valitsesid nad koalitsioonis PDS-iga .",seni valitsesid nad koalitsioonis PDS-iga,not_location,no
16,991744,ladvikus,ladvik,valitseb,valitsema,NaN,in,625087,""" Praegu valitseb ladvikus suhtumine : mõisa köis , las lohiseb . """,Praegu valitseb ladvikus suhtumine,not_location,no
17,1129733,rahastamises,rahastamine,valitseb,valitsema,NaN,in,709595,"Kõige palavam on endiste liiduvabariikide ning Ida-Euroopa maade filmitegijate jaoks kaos , mis valitseb postsotsialistlike maade filmitööstuse rahastamises , korraldamises ja seadusandluses .",mis valitseb rahastamises,not_location,no
18,1390798,Detsembris,detsember,valitsenud,valitsema,NaN,in,873298,Detsembris hooaja kaht esimest kiirlaskumist valitsenud Kostner viis pidupäeva väärika lõpuni .,Detsembris kiirlaskumist valitsenud,time,no
19,1473674,Situatsioonis,situatsioon,valitseb,valitsema,NaN,in,924765,"Situatsioonis kus praegu valitseb Eesti internetimaastikul suur segadus ning esitatavad keskkondade külastatavuse andmed on suuresti ebausaldusväärsed , siis HitBox Enterprise ?",Situatsioonis praegu valitseb internetimaastikul segadus,state,no


In [49]:
def chunks(lst, size=10):
    """Yield successive chunks of size N."""
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

In [117]:
results = []
results2 = []
responses = []
explanations = []
explanations_all = {}

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
# kui suure osa võtta "yes" vastustest "why" küsimusse
yes_subset_ratio = 0.2
batch_start_index = 0

batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in chunks(rows, size=bs):
    batch = []
    for ex in df_batch:
        batch.append({"l": ex["sentence"], "c": ex["head_form"]})

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    # võtab välja kõik batchis olnud "no" ja mõne "yes" ja küsib why
    """
    expl_response, batch_explanations, answered_idx = explain_non_locations(
            batch=batch,
            yes_no_results=result_yesno,
            yes_subset_ratio=yes_subset_ratio
        )

    # mapping: vastused õige lause+fraasiga kokku
    if batch_explanations is not None:
        used_tokens += expl_response.usage.total_tokens
        
        explanations.append(json.loads(batch_explanations))
        
        # Map batch-local -> global indices
        for local_i, explanation in json.loads(batch_explanations).items():
            global_i = batch_start_index + int(local_i)
            explanations_all[global_i] = explanation

    batch_start_index += len(batch)
    """
    if used_tokens >= 140000:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    
    break


classify 10


In [118]:
# batch 10 -> 2643 kuni 2900 tokenit (tavalise json kujul few-shotiga läheb u 4200 tokenit)

used_tokens # ümardame 580 lauset batchiga 10 on u 134K tokenit responsi põhjal

2879

In [119]:
type(results[0])

dict

In [120]:
results

[{'a': 'no'},
 {'a': 'yes'},
 {'a': 'yes'},
 {'a': 'no'},
 {'a': 'no'},
 {'a': 'no'},
 {'a': 'yes'},
 {'a': 'no'},
 {'a': 'no'},
 {'a': 'no'}]

### kui küsida mitut vastust (yes/no, lühike ja pikk vastus)

In [42]:
answers = []
shorts = []
long = []

problematic = []

bs = 10

for elem in results:
    try:
        #print(len(elem.split("\n")))
        if isinstance(elem, str):
            data = json.loads(elem)
            if len(data) != bs:
                for i in range(bs):
                    answers.append("?")
                    shorts.append("?")
                    long.append("?")
                problematic.append(data)
            else:
                for item in data:
                    answers.append(item["a"])
                    shorts.append(item["s"])
                    long.append(item["r"])
        elif isinstance(elem, dict):
            if "a" in elem.keys() and "s" in elem.keys() and "r" in elem.keys():
                answers.append(elem["a"])
                shorts.append(elem["s"])
                long.append(elem["r"])
            else:
                raise Exception("Missing keys!")
    except Exception as e:
        #print(len(elem.split("\n")))
        print(elem)
        #continue

### kui küsida ainult a:yes/no

In [124]:
answers = []

problematic = []

bs = 10

for elem in results:
    try:
        #print(len(elem.split("\n")))
        if isinstance(elem, str):
            data = json.loads(elem)
            if len(data) != bs:
                for i in range(bs):
                    answers.append("?")
                problematic.append(data)
            else:
                for item in data:
                    answers.append(item["a"])
        elif isinstance(elem, dict):
            if "a" in elem.keys():
                answers.append(elem["a"])
            else:
                raise Exception("Missing keys!")
    except Exception as e:
        #print(len(elem.split("\n")))
        print(elem)
        #continue

In [125]:
df["gpt_is_loc"] = answers
#df["short_answ"] = shorts
#df["long_answ"] = long

/tmp/ipykernel_21735/257831460.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["gpt_is_loc"] = answers


In [126]:
df

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase,ekilex_tag,eki_is_loc,gpt_is_loc
10,122893,veebruaris,veebruar,valitses,valitsema,NaN,in,71692,"1926. aasta veebruaris , kui Helen Wills saabus Cannes'i , et mängida tennist Suzanne Lengleniga , valitses Rivieras postimpressionistlik õhkkond .",veebruaris saabus valitses Rivieras õhkkond,time,no,no
11,178657,ladvikus,ladvik,valitsenud,valitsema,NaN,in,106005,8. juulil tekkis kaitsejõudude ladvikus valitsenud segaduse tõttu jäägritel vastasseis Vene eriüksusega .,ladvikus valitsenud,not_location,no,yes
12,186045,halduskogudes,halduskogu,valitseb,valitsema,NaN,in,110693,"Ka aedlinnaosade halduskogudes valitseb meelsus , mille järgi tuleks nii põhikrunt kui reservmaa lugeda ühtseks hoone teenindamiseks vajalikuks maaks .",halduskogudes valitseb meelsus,not_location,no,yes
13,304611,suhtumises,suhtumine,valitseb,valitsema,NaN,in,183264,Terav kontrast valitseb ka suhtumises lähimasse läänenaabrisse .,kontrast valitseb suhtumises,not_location,no,no
14,347538,teadvuses,teadvus,valitses,valitsema,NaN,in,213063,"Kuna vadja keelel puudus aga juba siis väljund ametlikku asjaajamismaailma ( ja tänapäevaks puudub parku juba täielikult igasugune sotsiaalne mõõde ) , valitses Ariste informantide teadvuses valdavalt vene-vadja kaksik- või lausa vene-ingeri-vadja kolmikidentiteet , mis väljaspool Ariste ja ta õpilastega suhtlussituatsioone ka valdas ja igapäevaeluliselt valdama jäi .",puudus valitses teadvuses valdavalt kaksik- kolmikidentiteet,not_location,no,no
15,692133,koalitsioonis,koalitsioon,valitsesid,valitsema,NaN,in,442427,"CDU kogus absoluutse häälte-enamuse ning hakkab seal nüüd üksi valitsema , seni valitsesid nad koalitsioonis PDS-iga .",seni valitsesid nad koalitsioonis PDS-iga,not_location,no,no
16,991744,ladvikus,ladvik,valitseb,valitsema,NaN,in,625087,""" Praegu valitseb ladvikus suhtumine : mõisa köis , las lohiseb . """,Praegu valitseb ladvikus suhtumine,not_location,no,yes
17,1129733,rahastamises,rahastamine,valitseb,valitsema,NaN,in,709595,"Kõige palavam on endiste liiduvabariikide ning Ida-Euroopa maade filmitegijate jaoks kaos , mis valitseb postsotsialistlike maade filmitööstuse rahastamises , korraldamises ja seadusandluses .",mis valitseb rahastamises,not_location,no,no
18,1390798,Detsembris,detsember,valitsenud,valitsema,NaN,in,873298,Detsembris hooaja kaht esimest kiirlaskumist valitsenud Kostner viis pidupäeva väärika lõpuni .,Detsembris kiirlaskumist valitsenud,time,no,no
19,1473674,Situatsioonis,situatsioon,valitseb,valitsema,NaN,in,924765,"Situatsioonis kus praegu valitseb Eesti internetimaastikul suur segadus ning esitatavad keskkondade külastatavuse andmed on suuresti ebausaldusväärsed , siis HitBox Enterprise ?",Situatsioonis praegu valitseb internetimaastikul segadus,state,no,no


#### suhtumises, teadvuses, koalitsioonis, rahastamises, situatsioonis - kas peaks olema ka tegelikult kohad?